# Préparation SFT — MediQAl, questions ouvertes (oeq)

Questions ouvertes sur des cas cliniques, avec une **réponse rédigée** par des enseignants en médecine (CC-BY-4.0).
Elles apprennent au modèle à **expliquer** en français, ce que les QCM ne font pas.

- **Une ligne par cas clinique** : le cas, puis les questions dans leur ordre (`cc_question_number`), chacune
  suivie de sa réponse. Un cas trop long est coupé en tranches. À l'entraînement, **toutes les réponses** sont
  apprises (`assistant_only_loss`), chacune avec l'historique qui la précède.
- **Sélection pour le triage** : cas avec constantes ou vocabulaire d'urgence, dans un budget de questions
  (`projet_14.selection`).
- **Format commun** à tous les notebooks v4 : `projet_14.conversation`.
- **Sortie** : `data/v4/sft_mediqal_oeq_{train,validation,test}.jsonl`, puis la version anonymisée dans `data/v4/anonymized/`.

# Import

In [1]:
import re
from pathlib import Path

import pandas as pd
from datasets import Dataset, load_dataset

from projet_14.conversation import CASE_LABEL, conversation_windows, split_of
from projet_14.mediqal import case_id, clean_text
from projet_14.rgpd import run
from projet_14.selection import case_table, select_cases

OUT = Path("../data/v4")
SOURCE, STEM = "MediQAl-oeq", "sft_mediqal_oeq"
MAX_EXAMPLE_CHARS = 4_000  # ~1 200 tokens : sous le max_length du SFT (1 536)

# Sélection pour le triage (environ 3 000 questions en français : oeq 1 100, mcqu 1 200, mcqm 700)
BUDGET = 1_100            # questions retenues, tous splits confondus
MAX_SUBJECT_SHARE = 0.15  # part maximale du budget par spécialité

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement

MediQAl ne publie les questions ouvertes que dans une partie `test` : on refait nous-mêmes le découpage
train / validation / test, par cas clinique (plus bas).

In [2]:
ds = load_dataset("ANR-MALADES/MediQAl", "oeq")
df = ds["test"].to_pandas()
df.shape

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


(4969, 7)

# Analyse

In [3]:
print(df["medical_subject"].value_counts().head(10), "\n")
print(df["question_type"].value_counts(), "\n")
print(f"avec cas clinique : {df['clinical_case'].notna().mean():.0%}")
df.head()

medical_subject
Pharmacy                     1544
Pediatric Cardiology         1225
Cardiology                    217
Nephro-Urology                193
Pulmonology                   184
Intensive Care                154
Neurology                     150
Gynecology and Obstetrics     149
Otorhinolaryngology (ENT)     126
Hematology                    124
Name: count, dtype: int64 

question_type
Reasoning        3125
Understanding    1842
Raisonnement        2
Name: count, dtype: int64 

avec cas clinique : 80%


,id,clinical_case,cc_question_number,question,answer,medical_subject,question_type
0,1,"Homme, 58 ans, majoration de dyspnée chez un B...",1,Quels sont éléments de gravité au moment de l’...,"FC>125bpm\n Terrain : ATCD de BPCO, insuffisan...",Emergency Medicine,Reasoning
1,2,"Homme, 58 ans, majoration de dyspnée chez un B...",2,Caractérisez en une phrase le tableau syndrom...,Insuffisance respiratoire aiguë / exacerbation...,Emergency Medicine,Reasoning
2,3,"Homme, 58 ans, majoration de dyspnée chez un B...",3,Quelle est l'explication la plus probable de l...,"Signes d’hypercapnie, pas d’effet du traitemen...",Emergency Medicine,Reasoning
3,4,"Homme, 58 ans, majoration de dyspnée chez un B...",4,Où le patient doit-il être hospitalisé ?,"En réanimation, car pas de réponse au traiteme...",Emergency Medicine,Reasoning
4,5,"M. Shoot, 30 ans.\n Agitation à la sortie d’un...",1,Quelle substance a-t-il prise ce soir ? Quels ...,"Cocaïne, qui donne un état d’agitation.\n Sign...",Emergency Medicine,Reasoning


# Nettoyage

Espaces et lignes vides en trop, type de question harmonisé (`Raisonnement` → `Reasoning`), ordre de la question
dans son cas en nombre, doublons retirés.

In [4]:
for col in ("clinical_case", "question", "answer"):
    df[col] = df[col].map(clean_text, na_action="ignore")
df["clinical_case"] = df["clinical_case"].replace("", None)
df["question_type"] = df["question_type"].replace({"Raisonnement": "Reasoning"})
df["order"] = pd.to_numeric(df["cc_question_number"], errors="coerce")

df = df.dropna(subset=["question", "answer"]).drop_duplicates(subset=["clinical_case", "question"])
df.shape

(4950, 8)

# Filtres

- **Questions qui renvoient à une image ou un document** (« ci-dessous », « figure »…) : la réponse est impossible à
  partir du seul texte.
- **Spécialités exclues** : la cardiologie pédiatrique (un quart des questions) est loin des urgences adultes.

In [5]:
IMAGE = re.compile(r"ci-dessous|ci-contre|ci-joint|figure|l'image|cette image|photo|schéma suivant|document suivant|annexe", re.I)
EXCLUDED_SUBJECTS = ["Pediatric Cardiology"]

image = df["question"].str.contains(IMAGE) | df["clinical_case"].fillna("").str.contains(IMAGE)
excluded = df["medical_subject"].isin(EXCLUDED_SUBJECTS)
print(f"renvoient à une image ou un document : {image.sum()} ; spécialités exclues : {excluded.sum()}")
df = df[~image & ~excluded]
df.shape

renvoient à une image ou un document : 122 ; spécialités exclues : 1206


(3639, 8)

# Répartition par cas clinique

Toutes les questions d'un cas vont dans le même split (70 % train, 10 % validation, 20 % test), tiré de
l'identifiant du cas : le même cas tombe dans le même split dans tous les notebooks. Une question sans cas est
répartie seule.

In [6]:
key = df["clinical_case"].fillna(df["question"])  # le cas, ou la question elle-même s'il n'y a pas de cas
df["case_id"] = key.map(case_id)
df["split"] = key.map(split_of)

# Doublons à la casse ou aux espaces près (même question dans le même cas) : un seul gardé
question_norm = df["question"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
duplicate = pd.concat([df["case_id"], question_norm], axis=1).duplicated()
print(f"doublons à la casse ou aux espaces près : {duplicate.sum()}")
df = df[~duplicate]
pd.crosstab(df["split"], df["clinical_case"].notna().map({True: "avec cas", False: "sans cas"}), margins=True)

doublons à la casse ou aux espaces près : 0


clinical_case,avec cas,sans cas,All
split,,,
test,637,74,711
train,2366,224,2590
validation,292,46,338
All,3295,344,3639


# Conversations par cas

Les questions d'un cas sont triées dans leur ordre, et le cas devient **une ligne** : une conversation complète. Un
cas trop long est coupé en tranches consécutives ; chaque tranche reprend le cas clinique en tête.

In [7]:
rows = []
for cid, group in df.sort_values(["order", "id"]).groupby("case_id", sort=False):
    case = group["clinical_case"].iloc[0]
    case = case if isinstance(case, str) else None  # question sans cas (valeur manquante : NaN)
    turns = [{"question": r.question, "answer": r.answer} for r in group.itertuples()]
    for start, messages in conversation_windows(case, turns):
        part = group.iloc[start:start + (len(messages) - 1) // 2]  # questions de cette ligne
        rows.append({
            "ids": [str(i) for i in part["id"]], "source": SOURCE, "case_id": cid, "split": part["split"].iloc[0],
            "n_questions": len(part), "medical_subject": part["medical_subject"].iloc[0],
            "question_types": list(part["question_type"]), "lang": "fr", "messages": messages,
        })
examples = pd.DataFrame(rows)

# Une question seule déjà trop longue (cas très long) : écartée plutôt que coupée à l'entraînement
examples["chars"] = examples["messages"].map(lambda m: sum(len(x["content"]) for x in m))
too_long = examples["chars"] > MAX_EXAMPLE_CHARS
print(f"{len(examples)} lignes ({examples['n_questions'].sum()} questions), dont {too_long.sum()} trop longues écartées")
examples = examples[~too_long]
examples.groupby("split").agg(lignes=("n_questions", "size"), questions=("n_questions", "sum"))

1355 lignes (3639 questions), dont 7 trop longues écartées


,lignes,questions
split,,
test,285,710
train,932,2584
validation,131,338


In [8]:
# Une ligne à plusieurs questions, telle que la verra le modèle
example = examples[examples["n_questions"] >= 3].iloc[0]
for message in example["messages"]:
    print(f"--- {message['role']}\n{message['content'][:500]}\n")

--- system
Tu es un assistant médical expert. Réponds de façon précise et concise.

--- user
Cas clinique : Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.

Examen d’entrée :
Constantes :
- FC = 130bpm
- PA = 130/80 aux deux bras
- FR = 40/min
- SpO2 = 80%
- T = 38 ,5°C
Tirage sus-sternal
Cyanose
Sueurs
Auscultation :
- Ronchi bilatéraux et sibilants
- Tympanisme bilatéral
Hépatomégalie douloureuse
Reflux hépato-jugulaire
Turgescence jugulaire
Hippocratisme digital
Cyanose unguéale

ATCD
Fumeur (80 PA)
BPCO

Traitement habituel
β2 mimétiques

HDM : depuis 1 semain

--- assistant
FC>125bpm
Terrain : ATCD de BPCO, insuffisance respiratoire chronique (hippocratisme digital)
Signes de détresse respiratoire avec hypoxie et hypercapnie :
- SpO2<90%
- FR>30/min
- Sueurs
- Cyanose
- Tirage
Signes d’insuffisance droite
- Hépatomégalie douloureuse
- Reflux
- Turgescence

--- user
Question : Caractérisez en une phrase le  tableau syndromique de ce patient. Quelles en sont l

# Sélection pour le triage

Le corpus complet (~30 000 questions en français) dépasse le besoin : environ 5 000 paires SFT, dont ~3 000 en
français. On garde les **cas les plus proches du triage**, par cas entier (`projet_14.selection`) :

1. **Cas clinique obligatoire** : le triage part toujours d'un patient décrit.
2. **Score de pertinence** (0 à 2) : **constantes** dans le cas (PA, FC, SpO2, température, Glasgow…) +
   vocabulaire d'**urgence** ou de conduite à tenir (urgence, hospitalisation, gravité, choc, détresse…).
3. **Budget** de `BUDGET` questions, réparti entre les splits (70 / 10 / 20) : dans chaque split, cas pris par score
   décroissant (score ≥ 1), puis dans un ordre pseudo-aléatoire fixe tiré de l'identifiant du cas.
4. **Plafond par spécialité** : `MAX_SUBJECT_SHARE` du budget au plus, pour garder la diversité.

Le même tri s'applique aux trois splits : le test ressemble au train.

In [9]:
cases = case_table(examples, lang="fr", group="medical_subject")
# Questions disponibles selon la présence d'un cas et le score
print(pd.crosstab(cases["patient"].map({True: "avec cas", False: "sans cas"}), cases["score"],
                  values=cases["n_questions"], aggfunc="sum", margins=True).fillna(0).astype(int), "\n")

selected = select_cases(cases, BUDGET, MAX_SUBJECT_SHARE, group="medical_subject")
examples = examples[examples["case_id"].isin(selected["case_id"])]
print(f"retenus : {len(selected)} cas, {selected['n_questions'].sum()} questions (budget {BUDGET})")
selected.groupby("split").agg(cas=("case_id", "size"), questions=("n_questions", "sum"),
                              constantes=("constantes", "mean"), urgence=("urgence", "mean")).round(2)

score       0     1     2   All
patient                        
avec cas  616  1401  1271  3288
sans cas  326    18     0   344
All       942  1419  1271  3632 

retenus : 204 cas, 1098 questions (budget 1100)


,cas,questions,constantes,urgence
split,,,,
test,45,219,0.78,0.98
train,138,770,0.91,0.99
validation,21,109,0.67,0.90


In [10]:
# Spécialités retenues (questions) : aucune ne dépasse MAX_SUBJECT_SHARE du budget
selected.groupby("medical_subject")["n_questions"].sum().sort_values(ascending=False)

medical_subject
Pharmacy                            161
Intensive Care                      140
Pulmonology                         106
Cardiology                           85
Neurology                            67
Emergency Medicine                   64
Infectious Diseases                  63
Hepato-Gastroenterology              59
Hematology                           51
Nephro-Urology                       49
Gynecology and Obstetrics            39
Otorhinolaryngology (ENT)            34
Forensic Medicine and Toxicology     33
Orthopedics                          32
Pediatrics                           28
Dermatology                          21
Rheumatology                         15
Psychiatry                           15
Pharmacology                          9
Occupational Medicine                 8
Endocrinology and Metabolism          7
Ophthalmology                         6
Public Health                         6
Name: n_questions, dtype: int64

# Export JSONL

In [12]:
OUT.mkdir(parents=True, exist_ok=True)
METADATA = ["ids", "source", "case_id", "n_questions", "medical_subject", "question_types", "lang"]
for split in ("train", "validation", "test"):
    part = examples[examples["split"] == split]
    Dataset.from_pandas(part[METADATA + ["messages"]], preserve_index=False).to_json(
        OUT / f"sft_mediqal_oeq_{split}.jsonl", force_ascii=False)
    print(split, len(part), "lignes")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.85ba/s]

train 248 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 225.04ba/s]

validation 37 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 138.35ba/s]

test 84 lignes


# Anonymisation des données

Presidio sur les messages **user et assistant** : les cas décrivent des patients, et les réponses peuvent reprendre
leur nom (« M. Shoot, 30 ans »). Dans les réponses, seuls les noms placés après une civilité sont remplacés
(`title_only_roles`) : les autres détections y sont presque toutes des termes médicaux (médicaments, germes, éponymes).

In [13]:
reports = {
    split: run(OUT / f"sft_mediqal_oeq_{split}.jsonl", lang="fr", output_dir=OUT / "anonymized",
               roles=["user", "assistant"], title_only_roles=["assistant"])
    for split in ("train", "validation", "test")
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,248,147,392
validation,37,22,66
test,84,42,106


# Contrôle final

In [14]:
final = load_dataset("json", data_files={s: str(OUT / "anonymized" / f"sft_mediqal_oeq_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
print(final)

# Aucun cas clinique partagé entre train, validation et test
cases = {s: set(d["case_id"]) for s, d in final.items()}
assert not cases["train"] & (cases["validation"] | cases["test"]), "cas partagé entre train et évaluation"
assert not cases["validation"] & cases["test"], "cas partagé entre validation et test"
# Chaque ligne part d'un cas clinique (sélection pour le triage)
assert all(m[1]["content"].startswith(CASE_LABEL) for d in final.values() for m in d["messages"]), "ligne sans cas clinique"
print("OK : aucun cas partagé entre les splits, toutes les lignes partent d'un cas clinique")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 248 examples [00:00, 33065.91 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 37 examples [00:00, 6304.66 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 84 examples [00:00, 16802.82 examples/s]

DatasetDict({
    train: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'lang', 'messages', 'persons_replaced'],
        num_rows: 248
    })
    validation: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'lang', 'messages', 'persons_replaced'],
        num_rows: 37
    })
    test: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'lang', 'messages', 'persons_replaced'],
        num_rows: 84
    })
})
OK : aucun cas partagé entre les splits, toutes les lignes partent d'un cas clinique
